# 4. NinjaTrader 8 Implementation - ES 15min Mean Reversion Strategy
## Production-Ready Script for Prop Firm / Funded Account

This notebook contains the final NinjaTrader 8 implementation of the ES 15min Mean Reversion strategy.

**Strategy Parameters (Optimized & Validated):**
- Entry Threshold: 1.0% (price move)
- RSI(2) Threshold: 16 (oversold) / 84 (overbought)
- ATR Stop Multiplier: 1.4x
- ATR Target Multiplier: 3.0x
- Time Stop: 3:50 PM NY
- Volume Ratio: > 1.2x
- Day Filter: Tuesday-Friday
- Time Window: 9:30 AM - 3:30 PM NY

**Inheritance:** The strategy inherits from `HealthMonitor` (which inherits from `PerformanceReporter`), providing:
- Health monitoring (slippage, order rejects, frozen orders, partial fills, etc.)
- Performance reporting to backend
- Contract rollover alerts
- Inactivity alerts
- Connection status monitoring

**Key Features:**
- Dynamic ATR-based stop-loss and take-profit
- Time stop at 3:50 PM NY (no open positions after 4 PM)
- Volume confirmation
- RSI(2) oversold/overbought conditions
- Day and time filters
- Position sizing via UI parameter (Quantity)
- Real-time trade reporting to backend

In [ ]:
using System;
using System.ComponentModel.DataAnnotations;
using System.Windows.Media;
using NinjaTrader.NinjaScript.Indicators;
using NinjaTrader.NinjaScript.Strategies;
using NinjaTrader.Cbi;
using NinjaTrader.Data;

namespace NinjaTrader.NinjaScript.Strategies
{
    public class MRES15min : HealthMonitor
    {
        // ================================ INDICATORS ================================
        private RSI rsi2;
        private ATR atr14;
        private SMA volumeMA20;

        // ================================ STATE TRACKING ================================
        private bool hasPosition;
        private DateTime entryTime;
        private double entryPrice;
        private double atrValue;

        // ================================ INITIALIZATION ================================
        protected override void OnStateChange()
        {
            if (State == State.SetDefaults)
            {
                Name = "MRES15min";
                Description = "ES 15min Mean Reversion Strategy - Validated with PF 1.80, Sharpe 3.64";

                Quantity = 1;
                EntryThresholdPercent = 1.0;
                RsiThreshold = 16;
                AtrStopMultiplier = 1.4;
                AtrTargetMultiplier = 3.0;
                VolumeRatioThreshold = 1.2;
                TimeStopHour = 15;
                TimeStopMinute = 50;

                MaxBarsWithoutTrade = 50;
                EnableInactivityAlert = true;
                RolloverAlertDays = 5;

                Calculate = Calculate.OnBarClose;
                IsInstantiatedOnEachOptimizationIteration = false;
            }
            else if (State == State.DataLoaded)
            {
                rsi2 = RSI(2, 0);
                rsi2.Plots[0].Brush = Brushes.Magenta;
                AddChartIndicator(rsi2);

                atr14 = ATR(14);
                atr14.Plots[0].Brush = Brushes.Orange;
                AddChartIndicator(atr14);

                volumeMA20 = SMA(Volume, 20);
                volumeMA20.Plots[0].Brush = Brushes.Gray;
                volumeMA20.Plots[0].Width = 2;
                AddChartIndicator(volumeMA20);

                hasPosition = false;
                entryTime = DateTime.MinValue;
                entryPrice = 0;
                atrValue = 0;
            }

            base.OnStateChange();
        }

        // ================================ MAIN TRADING LOGIC ================================
        protected override void OnBarUpdate()
        {
            base.OnBarUpdate();

            if (CurrentBar < BarsRequiredToTrade) return;
            if (rsi2 == null || atr14 == null || volumeMA20 == null) return;
            if (rsi2[0] == 0 || double.IsNaN(rsi2[0])) return;
            if (atr14[0] == 0 || double.IsNaN(atr14[0])) return;
            if (volumeMA20[0] == 0 || double.IsNaN(volumeMA20[0])) return;

            DateTime now = Time[0];
            int hour = now.Hour;
            int minute = now.Minute;

            bool isWithinTimeWindow = (hour > 9 || (hour == 9 && minute >= 30)) && (hour < 15 || (hour == 15 && minute <= 30));
            bool isAllowedDay = (now.DayOfWeek != DayOfWeek.Monday && now.DayOfWeek != DayOfWeek.Saturday && now.DayOfWeek != DayOfWeek.Sunday);
            bool isTimeStop = (hour > TimeStopHour || (hour == TimeStopHour && minute >= TimeStopMinute));

            // ============================================================
            // 1. EXIT CONDITIONS
            // ============================================================
            if (Position.MarketPosition != MarketPosition.Flat)
            {
                if (isTimeStop)
                {
                    if (Position.MarketPosition == MarketPosition.Long)
                        ExitLong("TimeStop", "LongEntry");
                    else
                        ExitShort("TimeStop", "ShortEntry");
                    return;
                }
                return;
            }

            // ============================================================
            // 2. ENTRY CONDITIONS
            // ============================================================
            if (!isWithinTimeWindow || !isAllowedDay) return;

            double closePctChange = (Close[0] - Close[1]) / Close[1] * 100.0;
            double volumeRatio = Volume[0] / volumeMA20[0];
            double rsiValue = rsi2[0];
            atrValue = atr14[0];
            double entryThreshold = EntryThresholdPercent;

            // --- LONG ENTRY ---
            if (closePctChange < -entryThreshold &&
                rsiValue < RsiThreshold &&
                volumeRatio > VolumeRatioThreshold)
            {
                SetStopLoss(CalculationMode.Price, Close[0] - (atrValue * AtrStopMultiplier));
                SetProfitTarget(CalculationMode.Price, Close[0] + (atrValue * AtrTargetMultiplier));

                entryPrice = Close[0];
                entryTime = Time[0];
                hasPosition = true;
                EnterLong(Quantity, "LongEntry");
                Print($"📈 LONG Entry: Price={entryPrice:F2}, RSI={rsiValue:F2}, VolRatio={volumeRatio:F2}, Stop={Close[0] - (atrValue * AtrStopMultiplier):F2}, Target={Close[0] + (atrValue * AtrTargetMultiplier):F2}");
                return;
            }

            // --- SHORT ENTRY ---
            if (closePctChange > entryThreshold &&
                rsiValue > (100 - RsiThreshold) &&
                volumeRatio > VolumeRatioThreshold)
            {
                SetStopLoss(CalculationMode.Price, Close[0] + (atrValue * AtrStopMultiplier));
                SetProfitTarget(CalculationMode.Price, Close[0] - (atrValue * AtrTargetMultiplier));

                entryPrice = Close[0];
                entryTime = Time[0];
                hasPosition = true;
                EnterShort(Quantity, "ShortEntry");
                Print($"📉 SHORT Entry: Price={entryPrice:F2}, RSI={rsiValue:F2}, VolRatio={volumeRatio:F2}, Stop={Close[0] + (atrValue * AtrStopMultiplier):F2}, Target={Close[0] - (atrValue * AtrTargetMultiplier):F2}");
                return;
            }
        }

        // ============================================================
        // 3. POSITION CLOSING - RESET STATE
        // ============================================================
        protected override void OnPositionUpdate(Position position, double averagePrice, int quantity, MarketPosition marketPosition)
        {
            if (quantity == 0 && hasPosition)
            {
                hasPosition = false;
                entryPrice = 0;
                entryTime = DateTime.MinValue;
                atrValue = 0;

                SetStopLoss(CalculationMode.Ticks, 0);
                SetProfitTarget(CalculationMode.Ticks, 0);
            }

            base.OnPositionUpdate(position, averagePrice, quantity, marketPosition);
        }

        // ============================================================
        // 4. UI-EDITABLE PARAMETERS
        // ============================================================

        [Range(1, 100)]
        [NinjaScriptProperty]
        [Display(Name = "Quantity", Description = "Number of contracts per order", Order = 1, GroupName = "Order Management")]
        public int Quantity { get; set; }

        [Range(0.5, 3.0)]
        [NinjaScriptProperty]
        [Display(Name = "Entry Threshold (%)", Description = "Minimum price move % to trigger entry", Order = 2, GroupName = "Strategy Parameters")]
        public double EntryThresholdPercent { get; set; }

        [Range(10, 25)]
        [NinjaScriptProperty]
        [Display(Name = "RSI(2) Threshold", Description = "RSI(2) oversold/overbought threshold", Order = 3, GroupName = "Strategy Parameters")]
        public int RsiThreshold { get; set; }

        [Range(0.8, 2.0)]
        [NinjaScriptProperty]
        [Display(Name = "Volume Ratio Threshold", Description = "Minimum volume ratio vs 20-period average", Order = 4, GroupName = "Strategy Parameters")]
        public double VolumeRatioThreshold { get; set; }

        [Range(0.5, 3.0)]
        [NinjaScriptProperty]
        [Display(Name = "ATR Stop Multiplier", Description = "Stop loss = entry ± (ATR × multiplier)", Order = 5, GroupName = "Risk Management")]
        public double AtrStopMultiplier { get; set; }

        [Range(1.5, 5.0)]
        [NinjaScriptProperty]
        [Display(Name = "ATR Target Multiplier", Description = "Take profit = entry ± (ATR × multiplier)", Order = 6, GroupName = "Risk Management")]
        public double AtrTargetMultiplier { get; set; }

        [Range(0, 23)]
        [NinjaScriptProperty]
        [Display(Name = "Time Stop Hour", Description = "Hour (NY time) to force close all positions", Order = 7, GroupName = "Time Management")]
        public int TimeStopHour { get; set; }

        [Range(0, 59)]
        [NinjaScriptProperty]
        [Display(Name = "Time Stop Minute", Description = "Minute (NY time) to force close all positions", Order = 8, GroupName = "Time Management")]
        public int TimeStopMinute { get; set; }
    }
}

# 5. Deployment Instructions

## File Placement
1. Copy `MR_15min_MeanReversion.cs` to: Documents\NinjaTrader 8\bin\Custom\Strategies\
2. Compile in NinjaTrader:
- Go to **New → NinjaScript Editor**
- Right-click → **Compile**

## Strategy Setup
1. In NinjaTrader, open a chart with **ES 15-min** bars.
2. Right-click → **Strategies** → **Add Strategy**.
3. Select **MR_15min_MeanReversion**.
4. Configure parameters:
- **Quantity**: 1 (or adjust for your risk tolerance)
- **Entry Threshold (%)**: 1.0
- **RSI(2) Threshold**: 16
- **Volume Ratio Threshold**: 1.2
- **ATR Stop Multiplier**: 1.4
- **ATR Target Multiplier**: 3.0
- **Time Stop Hour**: 15
- **Time Stop Minute**: 50

5. **Backend URL**: Leave as configured (your monitoring backend).
6. **Auth Token**: Leave as configured.

## Health Monitoring (Inherited)
- **Max Bars Without Trade**: 50 (alerts if no trades)
- **Enable Inactivity Alert**: true
- **Rollover Alert (days)**: 5

## Risk Notes (for your $150k funded account)
- **1 contract** = ~$200 risk per trade (with 4-point stop)
- **Max loss streak of 7**: -$1,400 (26.7% of $5,250 drawdown limit)
- **Expected monthly return**: ~$400-$600 with 1 contract

## Next Steps
1. Test in **Simulation** mode for 3-6 months.
2. Monitor metrics via your backend dashboard.
3. After validation, enable in **Live** mode (start with 1 contract).
4. Add more strategies (trend following, etc.) to build a diversified portfolio.